# Embedding con nomic-embed-text-v1.5


In [ ]:
# 1. Verifica che la GPU sia attiva
import torch

print("CUDA disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\nATTENZIONE: nessuna GPU.")
    print("Runtime > Cambia tipo di runtime > GPU (T4), poi riesegui.")

In [ ]:
# 2. Dipendenze (einops serve al codice custom di nomic)
!pip install -q sentence-transformers einops

In [ ]:
# 3. Carica colab_input.parquet
from google.colab import files
import pandas as pd

caricati = files.upload()
nome = next(iter(caricati))
df = pd.read_parquet(nome)

print(f"{len(df)} unita'  |  {df['root'].nunique()} thread distinti")
df.head(3)

In [ ]:
# 4. Carica il modello sulla GPU
from sentence_transformers import SentenceTransformer

MODELLO = "nomic-ai/nomic-embed-text-v1.5"

PREFISSO = "clustering: "

m = SentenceTransformer(MODELLO, trust_remote_code=True, device="cuda")
print("finestra di contesto:", m.max_seq_length, "token")
print("dimensione embedding:", m.get_sentence_embedding_dimension())

In [ ]:
# 5. Quanto troncamento resta?
import numpy as np
import json

tok = m.tokenizer
testi_sfid = [PREFISSO + " ".join(json.loads(t)) for t in df["testi_json"]]
op_unici = df["op_testo"].drop_duplicates()
testi_op_unici = [PREFISSO + t for t in op_unici]

for nome_g, testi in (("sfidanti", testi_sfid), ("post OP", testi_op_unici)):
    n = np.array([len(tok.encode(t, add_special_tokens=True)) for t in testi])
    print(f"{nome_g:<10} mediana {np.median(n):>6.0f}  max {n.max():>6.0f}  "
          f"troncati {(n > m.max_seq_length).mean():.1%}")

In [ ]:
# 6. Similarita' con il post dell'OP

import time

t0 = time.time()
e_sfid = m.encode(testi_sfid, batch_size=32, convert_to_numpy=True,
                  normalize_embeddings=True, show_progress_bar=True)
print(f"sfidanti: {time.time()-t0:.0f}s")

t0 = time.time()
e_op_unici = m.encode(testi_op_unici, batch_size=32, convert_to_numpy=True,
                      normalize_embeddings=True, show_progress_bar=True)
print(f"post OP:  {time.time()-t0:.0f}s")

indice = {t: i for i, t in enumerate(op_unici)}
e_op = e_op_unici[[indice[t] for t in df["op_testo"]]]

# embedding normalizzati: il prodotto scalare E' la similarita' coseno
sim_op = (e_sfid * e_op).sum(axis=1)
print(f"\nsim media {sim_op.mean():.3f}  sd {sim_op.std():.3f}")

In [ ]:
# 7. Auto-similarita' fra repliche successive dello stesso sfidante

piatti, appartenenza = [], []
for i, t in enumerate(df["testi_json"]):
    parti = json.loads(t)
    if len(parti) < 2:
        continue
    for p in parti:
        piatti.append(PREFISSO + p)
        appartenenza.append(i)

print(f"{len(piatti)} commenti da codificare, "
      f"per {len(set(appartenenza))} unita' multi-commento")

e_parti = m.encode(piatti, batch_size=32, convert_to_numpy=True,
                   normalize_embeddings=True, show_progress_bar=True)

auto = np.full(len(df), np.nan)
appartenenza = np.array(appartenenza)
for i in np.unique(appartenenza):
    e = e_parti[appartenenza == i]
    auto[i] = float(np.mean([e[j] @ e[j + 1] for j in range(len(e) - 1)]))

print(f"auto-similarita' definita per {np.isfinite(auto).sum()} unita'")

In [ ]:
# 8. Controllo di sanita': lo sfidante e' piu' simile all'OP del PROPRIO

rng = np.random.default_rng(42)
perm = rng.permutation(len(df))
diverso = df["root"].to_numpy() != df["root"].to_numpy()[perm]
sim_caso = (e_sfid[diverso] * e_op[perm][diverso]).sum(axis=1)

print(f"coseno con il PROPRIO OP : {sim_op.mean():+.3f}")
print(f"coseno con un OP A CASO  : {sim_caso.mean():+.3f}")
print(f"scarto                   : {sim_op.mean() - sim_caso.mean():+.3f}")
print("\n(se lo scarto fosse ~0 il modello non distinguerebbe: sarebbe rotto)")

In [ ]:
# 9. Salva e scarica

out = pd.DataFrame({
    "unit_id": df["unit_id"],
    "sim_semantica_op_nomic": sim_op,
    "auto_similarita_nomic": auto,
})
out.to_parquet("nomic_semantic.parquet", index=False)
print(out.describe())

files.download("nomic_semantic.parquet")
print("\nSalva il file in data/ del progetto, poi esegui:")
print("  venv/Scripts/python.exe src/pipeline/step5c_ablation_nomic.py")